# QUERY 2
Nombre de banco, cuenta de origen y monto de la max. transacción USD de cada banco.

In [3]:
# importamos dataset
import pandas as pd

CANTIDAD_FILAS = 100000
RUTA_DE_DATASETS = "../../datasets"
NOMBRE_ARCHIVO_TRANSACCIONES = "LI-Small_Trans.csv"
NOMBRE_ARCHIVO_CUENTAS = "LI-Small_accounts.csv"
RUTA_TRANSACCIONES_SAMPLE = f"{RUTA_DE_DATASETS}/transacciones_sample.csv"
RUTA_CUENTAS_SAMPLE = f"{RUTA_DE_DATASETS}/accounts_sample.csv"

transacciones_completas = pd.read_csv(
    f"{RUTA_DE_DATASETS}/{NOMBRE_ARCHIVO_TRANSACCIONES}",
    dtype={
        "From Bank": "string",
        "Account": "string",
        "To Bank": "string",
        "Account.1": "string"
    }
)
cuentas_completas = pd.read_csv(
    f"{RUTA_DE_DATASETS}/{NOMBRE_ARCHIVO_CUENTAS}",
    dtype={
        "Bank ID": "string",
        "Account Number": "string"
    }
)
columnas_cuentas_originales = cuentas_completas.columns.tolist()

def normalizar_bank_id(serie):
    return serie.str.strip().str.lstrip("0").replace("", "0")

transacciones = transacciones_completas.sample(n=CANTIDAD_FILAS, random_state=42)
guardar_csv = lambda df, nombre_archivo: df.to_csv(nombre_archivo, index=False)
guardar_csv(transacciones, RUTA_TRANSACCIONES_SAMPLE)

transacciones_sample = pd.read_csv(
    RUTA_TRANSACCIONES_SAMPLE,
    dtype={
        "From Bank": "string",
        "Account": "string",
        "To Bank": "string",
        "Account.1": "string"
    }
)

transacciones_sample["From Bank Normalized"] = normalizar_bank_id(transacciones_sample["From Bank"])
transacciones_sample["To Bank Normalized"] = normalizar_bank_id(transacciones_sample["To Bank"])
cuentas_completas["Bank ID Normalized"] = normalizar_bank_id(cuentas_completas["Bank ID"])

cuentas_relevantes = pd.concat([
    transacciones_sample[["From Bank Normalized", "Account"]].rename(
        columns={"From Bank Normalized": "Bank ID Normalized", "Account": "Account Number"}
    ),
    transacciones_sample[["To Bank Normalized", "Account.1"]].rename(
        columns={"To Bank Normalized": "Bank ID Normalized", "Account.1": "Account Number"}
    )
], ignore_index=True).dropna().drop_duplicates()

cuentas_sample = cuentas_completas.merge(
    cuentas_relevantes,
    on=["Bank ID Normalized", "Account Number"],
    how="inner"
)[columnas_cuentas_originales]
guardar_csv(cuentas_sample, RUTA_CUENTAS_SAMPLE)

cuentas_sample.head()

,Bank Name,Bank ID,Account Number,Entity ID,Entity Name
0,India Bank #19,123651,80ED98180,800F49110,Sole Proprietorship #43351
1,Portugal Bank #42,24029,80A36EE80,800964DC0,Partnership #51172
2,Bank of Topeka,25259,80CA95580,8005F8A20,Corporation #16457
3,Savings Bank of Madison,11,800D80580,800074010,Partnership #70833
4,Flagstone Federal Bank,16263,80CD7C390,8003617E0,Partnership #50770


In [4]:
RUTA_RESULTADO_QUERY2 = f"{RUTA_DE_DATASETS}/query2_result.csv"

transacciones_sample = pd.read_csv(
    RUTA_TRANSACCIONES_SAMPLE,
    dtype={
        "From Bank": "string",
        "Account": "string",
        "To Bank": "string",
        "Account.1": "string"
    }
)
cuentas_sample = pd.read_csv(
    RUTA_CUENTAS_SAMPLE,
    dtype={
        "Bank ID": "string",
        "Account Number": "string"
    }
)

transacciones_usd = transacciones_sample[
    transacciones_sample["Payment Currency"] == "US Dollar"
].copy()
transacciones_usd["From Bank Normalized"] = normalizar_bank_id(transacciones_usd["From Bank"])
transacciones_usd["Amount Paid"] = pd.to_numeric(transacciones_usd["Amount Paid"], errors="coerce")
transacciones_usd = transacciones_usd.dropna(subset=["Amount Paid"])

# Unir con nombres de banco antes de agrupar
bancos = cuentas_sample[["Bank ID", "Bank Name"]].copy()
bancos["From Bank Normalized"] = normalizar_bank_id(bancos["Bank ID"])
bancos = bancos[["From Bank Normalized", "Bank Name"]].drop_duplicates(subset=["From Bank Normalized"])

transacciones_con_banco = transacciones_usd.merge(bancos, on="From Bank Normalized", how="inner")

# Agrupar por Bank Name: un resultado por banco con el monto máximo y la cuenta que lo generó
idx_maximos = transacciones_con_banco.groupby("Bank Name")["Amount Paid"].idxmax()
resultado_query2 = transacciones_con_banco.loc[idx_maximos, ["Bank Name", "Account", "Amount Paid"]].rename(columns={
    "Account": "Cuenta Origen",
    "Amount Paid": "Monto Max USD"
}).sort_values(by="Bank Name").reset_index(drop=True)

guardar_csv(resultado_query2, RUTA_RESULTADO_QUERY2)
resultado_query2.head()

,Bank Name,Cuenta Origen,Monto Max USD
0,Acme Community Bank,80D4B3200,2233120.00
1,Acme Cooperative Bank,805CF65C0,3754.87
2,Acme Credit Union,803E98810,4181910.56
3,Acme Federal Bank,819AB5420,8120.47
4,Acme Savings Bank,80B7BF6D0,20651.30
